In [1]:
# ----------------------------------------------------------------------------
# 1. IMPORTS
# ----------------------------------------------------------------------------
import pandas as pd
import numpy as np
from datetime import datetime

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Feature Engineering
from sklearn.cluster import KMeans

# Hyper Params Tuning
from sklearn.model_selection import GridSearchCV

In [2]:
# ----------------------------------------------------------------------------
# 2. CARREGAR DADOS
# ----------------------------------------------------------------------------

dfTrain = pd.read_csv('training_data.csv', encoding='latin1')
dfTest = pd.read_csv('test_data.csv', encoding='latin1')

In [3]:
dfTrain.describe()

,AVERAGE_FREE_FLOW_SPEED,AVERAGE_TIME_DIFF,AVERAGE_FREE_FLOW_TIME,AVERAGE_TEMPERATURE,AVERAGE_ATMOSP_PRESSURE,AVERAGE_HUMIDITY,AVERAGE_WIND_SPEED,AVERAGE_PRECIPITATION
count,6812.000000,6812.000000,6812.000000,6812.000000,6812.000000,6812.000000,6812.000000,6812.0
mean,40.661010,25.637111,81.143952,16.193482,1017.388139,80.084190,3.058573,0.0
std,4.119023,33.510507,8.294401,5.163492,5.751061,18.238863,2.138421,0.0
min,30.500000,0.000000,46.400000,0.000000,985.000000,14.000000,0.000000,0.0
25%,37.600000,2.275000,75.400000,13.000000,1015.000000,69.750000,1.000000,0.0
50%,40.700000,12.200000,82.400000,16.000000,1017.000000,83.000000,3.000000,0.0
75%,43.500000,36.200000,87.400000,19.000000,1021.000000,93.000000,4.000000,0.0
max,55.900000,296.500000,112.000000,35.000000,1033.000000,100.000000,14.000000,0.0


In [4]:
dfTest.shape

(1500, 13)

In [5]:
dfTrain.head()

,city_name,record_date,AVERAGE_SPEED_DIFF,AVERAGE_FREE_FLOW_SPEED,AVERAGE_TIME_DIFF,AVERAGE_FREE_FLOW_TIME,LUMINOSITY,AVERAGE_TEMPERATURE,AVERAGE_ATMOSP_PRESSURE,AVERAGE_HUMIDITY,AVERAGE_WIND_SPEED,AVERAGE_CLOUDINESS,AVERAGE_PRECIPITATION,AVERAGE_RAIN
0,Porto,2019-08-29 07:00:00,Medium,41.5,11.5,71.4,LIGHT,15.0,1019.0,100.0,3.0,NaN,0.0,NaN
1,Porto,2018-08-10 14:00:00,High,41.7,48.3,87.4,LIGHT,21.0,1021.0,53.0,5.0,céu claro,0.0,NaN
2,Porto,2019-09-01 16:00:00,High,38.6,38.4,85.2,LIGHT,26.0,1014.0,61.0,4.0,NaN,0.0,NaN
3,Porto,2019-02-26 11:00:00,High,37.4,61.0,94.1,LIGHT,18.0,1025.0,48.0,4.0,céu claro,0.0,NaN
4,Porto,2019-06-06 12:00:00,Medium,41.6,50.4,77.0,LIGHT,15.0,1008.0,82.0,10.0,NaN,0.0,NaN


In [6]:
dfTest.head()

,city_name,record_date,AVERAGE_FREE_FLOW_SPEED,AVERAGE_TIME_DIFF,AVERAGE_FREE_FLOW_TIME,LUMINOSITY,AVERAGE_TEMPERATURE,AVERAGE_ATMOSP_PRESSURE,AVERAGE_HUMIDITY,AVERAGE_WIND_SPEED,AVERAGE_CLOUDINESS,AVERAGE_PRECIPITATION,AVERAGE_RAIN
0,Porto,2019-02-13 23:00:00,39.2,0.0,91.0,DARK,8.0,1026.0,71.0,1.0,céu claro,0.0,NaN
1,Porto,2018-11-28 20:00:00,42.5,12.2,76.8,DARK,11.0,1020.0,93.0,4.0,nuvens dispersas,0.0,NaN
2,Porto,2018-08-14 05:00:00,45.9,0.0,86.3,DARK,14.0,1017.0,93.0,0.0,NaN,0.0,NaN
3,Porto,2019-07-06 17:00:00,33.2,51.7,89.9,LIGHT,22.0,1016.0,77.0,4.0,céu pouco nublado,0.0,NaN
4,Porto,2018-10-15 06:00:00,44.0,3.5,85.5,DARK,12.0,1004.0,100.0,9.0,NaN,0.0,chuva fraca


In [7]:
dfTest.duplicated().value_counts()

False    1500
Name: count, dtype: int64

In [8]:
dfTrain.duplicated().value_counts()

False    6812
Name: count, dtype: int64

In [9]:
dfTrain['AVERAGE_SPEED_DIFF'].value_counts()

AVERAGE_SPEED_DIFF
Medium       1651
Low          1419
High         1063
Very_High     479
Name: count, dtype: int64

In [10]:
dfTrain['LUMINOSITY'].value_counts()

LUMINOSITY
LIGHT        3293
DARK         3253
LOW_LIGHT     266
Name: count, dtype: int64

In [11]:
dfTrain['AVERAGE_CLOUDINESS'].value_counts()

AVERAGE_CLOUDINESS
céu claro            1582
céu pouco nublado     516
nuvens dispersas      459
nuvens quebrados      448
algumas nuvens        422
nuvens quebradas      416
céu limpo             153
tempo nublado          67
nublado                67
Name: count, dtype: int64

In [12]:
dfTrain['AVERAGE_RAIN'].value_counts()

AVERAGE_RAIN
chuva fraca                    261
chuva moderada                 153
chuva leve                      45
aguaceiros fracos               38
chuva                           30
aguaceiros                      11
chuva forte                      8
trovoada com chuva leve          7
chuvisco fraco                   5
chuva de intensidade pesado      2
chuva de intensidade pesada      1
trovoada com chuva               1
chuvisco e chuva fraca           1
Name: count, dtype: int64

In [13]:
dfTrain['AVERAGE_PRECIPITATION'].value_counts()

AVERAGE_PRECIPITATION
0.0    6812
Name: count, dtype: int64

In [14]:
dfTrain.isna().sum()

city_name                     0
record_date                   0
AVERAGE_SPEED_DIFF         2200
AVERAGE_FREE_FLOW_SPEED       0
AVERAGE_TIME_DIFF             0
AVERAGE_FREE_FLOW_TIME        0
LUMINOSITY                    0
AVERAGE_TEMPERATURE           0
AVERAGE_ATMOSP_PRESSURE       0
AVERAGE_HUMIDITY              0
AVERAGE_WIND_SPEED            0
AVERAGE_CLOUDINESS         2682
AVERAGE_PRECIPITATION         0
AVERAGE_RAIN               6249
dtype: int64

In [15]:
dfTest.isna().sum()

city_name                     0
record_date                   0
AVERAGE_FREE_FLOW_SPEED       0
AVERAGE_TIME_DIFF             0
AVERAGE_FREE_FLOW_TIME        0
LUMINOSITY                    0
AVERAGE_TEMPERATURE           0
AVERAGE_ATMOSP_PRESSURE       0
AVERAGE_HUMIDITY              0
AVERAGE_WIND_SPEED            0
AVERAGE_CLOUDINESS          599
AVERAGE_PRECIPITATION         0
AVERAGE_RAIN               1360
dtype: int64

In [16]:
# ----------------------------------------------------------------------------
# 4. DROP DE COLUNAS
# ----------------------------------------------------------------------------
cols_to_drop = ['AVERAGE_PRECIPITATION', 'city_name']

dfTrain = dfTrain.drop(columns=[col for col in cols_to_drop if col in dfTrain.columns])
dfTest = dfTest.drop(columns=[col for col in cols_to_drop if col in dfTest.columns])

In [17]:
# Converter LUMINOSITY para 0/1 (DARK=0, LIGHT=1)
def convert_luminosity(df):
    df = df.copy()
    df['LUMINOSITY'] = df['LUMINOSITY'].map({'DARK': 0, 'LIGHT': 1, 'LOW_LIGHT': 2})
    return df

dfTrain = convert_luminosity(dfTrain)
dfTest = convert_luminosity(dfTest)

In [18]:
# ----------------------------------------------------------------------------
# 5. FEATURE ENGINEERING - DATAS
# ----------------------------------------------------------------------------
def extract_datetime_features(df):
    """Extrai features temporais da coluna record_date"""
    df = df.copy()
    
    # Converter para datetime
    df['record_date'] = pd.to_datetime(df['record_date'])
    
    # Extrair componentes
    df['month'] = df['record_date'].dt.month
    df['hour'] = df['record_date'].dt.hour
    df['day_of_week'] = df['record_date'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # Features adicionais
    df['is_rush_hour'] = ((df['hour'] >= 7.30) & (df['hour'] <= 9.30) | 
                          (df['hour'] >= 16.30) & (df['hour'] <= 20.30)).astype(int)
    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 6)).astype(int)
    df['day_period'] = pd.cut(df['hour'], bins=[0, 6, 12, 18, 24], 
                               labels=['night', 'morning', 'afternoon', 'evening'],
                               include_lowest=True)
    
    return df

dfTrain = extract_datetime_features(dfTrain)
dfTest = extract_datetime_features(dfTest)

In [19]:
# ----------------------------------------------------------------------------
# 6. TRATAMENTO DE MISSING VALUES COM MEDIANA
# ----------------------------------------------------------------------------
def impute_with_median(df):
    df = df.copy()
    
    # Selecionar colunas numéricas
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            print(f" {median_val}")
            df[col] = df[col].fillna(median_val)
            print(f"   {col}: imputado com mediana = {median_val:.2f}")
    
    return df

dfTrain = impute_with_median(dfTrain)
dfTest = impute_with_median(dfTest)

In [20]:
# ----------------------------------------------------------------------------
# 7. TRATAMENTO DO TARGET (APENAS TRAIN)
# ----------------------------------------------------------------------------

# Remover registos sem target no treino
print(f"Registos antes: {len(dfTrain)}")
dfTrain = dfTrain[dfTrain['AVERAGE_SPEED_DIFF'].notna()].copy()
print(f"Registos após remoção de nulls no target: {len(dfTrain)}")

Registos antes: 6812
Registos após remoção de nulls no target: 4612


In [21]:
# Mapear target para valores ordinais
target_mapping = {
    'None': 0,
    'Low': 1,
    'Medium': 2,
    'High': 3,
    'Very_High': 4
}

mapa_cloudiness = {
    'céu limpo': 0,
    'algumas nuvens': 1,
    'nuvens dispersas': 2,
    'nuvens quebrados': 3,
    'tempo nublado': 4
}

dfTrain['target_encoded'] = dfTrain['AVERAGE_SPEED_DIFF'].map(target_mapping)

dfTrain['AVERAGE_CLOUDINESS'] = dfTrain['AVERAGE_CLOUDINESS'].map(mapa_cloudiness)
dfTest['AVERAGE_CLOUDINESS'] = dfTest['AVERAGE_CLOUDINESS'].map(mapa_cloudiness)

In [22]:
# ----------------------------------------------------------------------------
# 8. ENCODING DE VARIÁVEIS CATEGÓRICAS
# ----------------------------------------------------------------------------

# Day Period
le_period = LabelEncoder()


dfTrain['day_period_encoded'] = le_period.fit_transform(dfTrain['day_period'].astype(str))
dfTest['day_period_encoded'] = le_period.transform(dfTest['day_period'].astype(str))

dfTrain['AVERAGE_RAIN_encoded'] = le_period.fit_transform(dfTrain['AVERAGE_RAIN'].astype(str))

In [23]:
# ----------------------------------------------------------------------------
# 9. CLUSTERING
# ----------------------------------------------------------------------------

cluster_features = ['AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'hour']

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

In [24]:
dfTrain['cluster'] = kmeans.fit_predict(dfTrain[cluster_features])
dfTest['cluster'] = kmeans.predict(dfTest[cluster_features])

In [25]:
# ----------------------------------------------------------------------------
# 10. SELEÇÃO DE FEATURES PARA MODELAÇÃO (CORRIGIDO)
# ----------------------------------------------------------------------------

# Features para o modelo
# NOTA: Removemos AVERAGE_SPEED_DIFF (o alvo!)
# E corrigimos AVERAGE_RAIN para AVERAGE_RAIN_encoded
feature_cols = [
    'AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME',
    'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY',
    'AVERAGE_WIND_SPEED',
    
    # Temporais
    'hour', 'day_of_week', 'is_weekend',
    'is_rush_hour',
    
    # Encoded
    'LUMINOSITY',
    
    # Cluster
    'cluster'
]

# Verificar disponibilidade
# Esta linha agora irá funcionar e selecionar as colunas corretas
available_features = [col for col in feature_cols if col in dfTrain.columns and col in dfTest.columns]

print(f"Features a serem usadas: {available_features}")

# PREPARAR DADOS PARA TREINO
X_train = dfTrain[available_features].copy()

# <-- CORREÇÃO AQUI
# O alvo (y) é a coluna que você codificou na célula [20]
y_train = dfTrain['target_encoded'].copy()

# PREPARAR DADOS PARA TESTE (SEM TARGET!)
X_test = dfTest[available_features].copy()

Features a serem usadas: ['AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME', 'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY', 'AVERAGE_WIND_SPEED', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'LUMINOSITY', 'cluster']


In [26]:
dfTrain.describe()

,record_date,AVERAGE_FREE_FLOW_SPEED,AVERAGE_TIME_DIFF,AVERAGE_FREE_FLOW_TIME,LUMINOSITY,AVERAGE_TEMPERATURE,AVERAGE_ATMOSP_PRESSURE,AVERAGE_HUMIDITY,AVERAGE_WIND_SPEED,AVERAGE_CLOUDINESS,month,hour,day_of_week,is_weekend,is_rush_hour,is_night,target_encoded,day_period_encoded,AVERAGE_RAIN_encoded,cluster
count,4612,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000,1021.000000,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000,4612.000000
mean,2019-03-06 17:59:23.313096192,40.082003,37.106288,81.591761,0.749350,17.293148,1017.374241,77.087056,3.361882,1.872674,7.074154,13.072203,2.959237,0.268213,0.327624,0.179965,2.130529,1.205117,10.565915,0.658500
min,2018-07-24 15:00:00,30.500000,0.600000,46.400000,0.000000,0.000000,985.000000,14.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,2018-10-30 11:45:00,37.100000,11.600000,76.100000,0.000000,14.000000,1015.000000,67.000000,2.000000,1.000000,5.000000,9.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,11.000000,0.000000
50%,2019-03-04 16:00:00,40.000000,25.600000,82.600000,1.000000,17.000000,1017.000000,82.000000,3.000000,2.000000,8.000000,13.000000,3.000000,0.000000,0.000000,0.000000,2.000000,1.000000,11.000000,0.000000
75%,2019-07-02 04:00:00,43.000000,50.925000,87.700000,1.000000,21.000000,1021.000000,93.000000,5.000000,3.000000,9.000000,18.000000,5.000000,1.000000,1.000000,0.000000,3.000000,2.000000,11.000000,2.000000
max,2019-09-30 18:00:00,55.900000,296.500000,112.000000,2.000000,35.000000,1033.000000,100.000000,13.000000,4.000000,12.000000,23.000000,6.000000,1.000000,1.000000,1.000000,4.000000,3.000000,12.000000,2.000000
std,NaN,4.256414,35.333911,8.260256,0.524037,5.205372,5.702105,18.700445,2.111047,1.001689,2.897611,5.647549,1.961588,0.443077,0.469398,0.384200,0.967866,1.071097,1.554683,0.882753


In [27]:
dfTest.describe()

,record_date,AVERAGE_FREE_FLOW_SPEED,AVERAGE_TIME_DIFF,AVERAGE_FREE_FLOW_TIME,LUMINOSITY,AVERAGE_TEMPERATURE,AVERAGE_ATMOSP_PRESSURE,AVERAGE_HUMIDITY,AVERAGE_WIND_SPEED,AVERAGE_CLOUDINESS,month,hour,day_of_week,is_weekend,is_rush_hour,is_night,day_period_encoded,cluster
count,1500,1500.0000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,335.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000
mean,2019-02-23 09:18:26.400000,40.8304,26.750533,81.194333,0.543333,16.104000,1017.457333,80.734000,3.116667,1.952239,7.105333,11.211333,2.984000,0.284000,0.235333,0.388000,1.618000,0.483333
min,2018-07-24 18:00:00,31.0000,0.000000,48.100000,0.000000,1.000000,985.000000,14.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2018-10-20 19:30:00,37.5000,2.500000,75.800000,0.000000,13.000000,1015.000000,72.000000,1.000000,1.000000,5.000000,5.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2019-02-15 17:00:00,41.0000,12.600000,82.300000,1.000000,16.000000,1018.000000,86.000000,3.000000,2.000000,8.000000,11.000000,3.000000,0.000000,0.000000,0.000000,2.000000,0.000000
75%,2019-06-25 00:15:00,43.9000,40.225000,87.600000,1.000000,19.000000,1021.000000,93.000000,4.000000,3.000000,9.000000,17.000000,5.000000,1.000000,0.000000,1.000000,3.000000,1.000000
max,2019-09-30 19:00:00,56.2000,232.300000,106.100000,2.000000,32.000000,1033.000000,100.000000,13.000000,4.000000,12.000000,23.000000,6.000000,1.000000,1.000000,1.000000,3.000000,2.000000
std,NaN,4.2396,34.089866,8.189691,0.555273,5.094293,5.840455,17.729358,2.198699,1.068375,2.979512,6.909305,1.994257,0.451087,0.424348,0.487457,1.167184,0.813734


In [28]:
# ----------------------------------------------------------------------------
# 11. LIMPEZA FINAL DE DADOS (APENAS INF)
# ----------------------------------------------------------------------------
# Substituir inf por NaN
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

In [32]:
# ----------------------------------------------------------------------------
# 12. SPLIT DE VALIDAÇÃO
# ----------------------------------------------------------------------------
param_grid_rf = {
    'n_estimators': [50,250, 300],
    'max_depth': [2, 3, 4],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [15,100,300],
}


X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, 
    test_size=0.3, 
    random_state=2020, 
    stratify=y_train
)

In [33]:
# ----------------------------------------------------------------------------
# 12.5. IMPUTAÇÃO DE VALORES EM FALTA (MEDIANA)
# ----------------------------------------------------------------------------

# Capturar os nomes das colunas antes que o imputer as transforme em array
feature_names = X_tr.columns

imputer = SimpleImputer(strategy='median')

# Fazer o 'fit' APENAS nos dados de treino (X_tr)
imputer.fit(X_tr) 

# Aplicar o 'transform' a todos os conjuntos
# O transform() retorna um array numpy, por isso reconstrói-se o DataFrame
X_tr = pd.DataFrame(imputer.transform(X_tr), columns=feature_names)
X_val = pd.DataFrame(imputer.transform(X_val), columns=feature_names)
X_test = pd.DataFrame(imputer.transform(X_test), columns=feature_names)

In [34]:
# ----------------------------------------------------------------------------
# 13. TREINO DOS MODELOS
# ----------------------------------------------------------------------------

# Instanciar o modelo base
rf_base = RandomForestClassifier(random_state=2020, n_jobs=-1)

# Criar o GridSearchCV
grid_search_rf = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid_rf,
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=2020),
    scoring='f1_weighted',
    verbose=2,
    n_jobs=-1
)

# Executar busca
grid_search_rf.fit(X_tr, y_tr)

# Melhor modelo
rf_model = grid_search_rf.best_estimator_
print(f"Melhor combinação de hiperparâmetros: {grid_search_rf.best_params_}")

# Avaliar no conjunto de validação
y_val_pred_rf = rf_model.predict(X_val)
acc_rf = accuracy_score(y_val, y_val_pred_rf)
f1_rf = f1_score(y_val, y_val_pred_rf, average='weighted')
print(f"   Accuracy: {acc_rf:.4f} | F1: {f1_rf:.4f}")


print("\nHistogram Gradient Boosting...")
gb_model = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.1,
    max_depth=4,
    min_samples_leaf=8,
    random_state=2020
)
gb_model.fit(X_tr, y_tr)
y_val_pred_gb = gb_model.predict(X_val)
acc_gb = accuracy_score(y_val, y_val_pred_gb)
f1_gb = f1_score(y_val, y_val_pred_gb, average='weighted')
print(f"   Accuracy: {acc_gb:.4f} | F1: {f1_gb:.4f}")

Fitting 3 folds for each of 54 candidates, totalling 162 fits
Melhor combinação de hiperparâmetros: {'max_depth': 4, 'min_samples_leaf': 15, 'min_samples_split': 10, 'n_estimators': 250}
   Accuracy: 0.7681 | F1: 0.7659

Histogram Gradient Boosting...
   Accuracy: 0.8136 | F1: 0.8131


In [35]:
# ----------------------------------------------------------------------------
# 14. RETREINAR NO DATASET COMPLETO

rf_model.fit(X_train, y_train)

,n_estimators,250
,criterion,'gini'
,max_depth,4
,min_samples_split,10
,min_samples_leaf,15
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [36]:
gb_model.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,max_iter,200
,max_leaf_nodes,31
,max_depth,4
,min_samples_leaf,8
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'
,monotonic_cst,None


In [37]:
# ----------------------------------------------------------------------------
# 15. PREDIÇÕES NO TEST SET (CALCULAR AVERAGE_SPEED_DIFF!)
# ----------------------------------------------------------------------------
# Predições
y_test_pred_rf = rf_model.predict(X_test)
y_test_pred_gb = gb_model.predict(X_test)
y_test_pred_ensemble = np.round((y_test_pred_rf + y_test_pred_gb) / 2).astype(int)

# Converter para labels originais
inverse_target_mapping = {v: k for k, v in target_mapping.items()}
predictions_rf = pd.Series(y_test_pred_rf).map(inverse_target_mapping)
predictions_gb = pd.Series(y_test_pred_gb).map(inverse_target_mapping)
predictions_ensemble = pd.Series(y_test_pred_ensemble).map(inverse_target_mapping)

print("\nDistribuição de Predições (Ensemble):")
print(predictions_ensemble.value_counts())


Distribuição de Predições (Ensemble):
Low          775
Medium       379
High         236
Very_High    110
Name: count, dtype: int64


In [38]:
# ----------------------------------------------------------------------------
# 16. GUARDAR OUTPUTS (FORMATO: RowId,Speed_Diff)
# ----------------------------------------------------------------------------
# Criar RowId sequencial começando de 0
row_ids = range(1, len(dfTest) + 1)

# Output 1: Random Forest
output1 = pd.DataFrame({
    'RowId': row_ids,
    'Speed_Diff': predictions_rf.values
})
output1.to_csv('output1.csv', index=False)

# Output 2: Gradient Boosting
output2 = pd.DataFrame({
    'RowId': row_ids,
    'Speed_Diff': predictions_gb.values
})
output2.to_csv('output2.csv', index=False)

# Output 3: Ensemble
output3 = pd.DataFrame({
    'RowId': row_ids,
    'Speed_Diff': predictions_ensemble.values
})
output3.to_csv('output3.csv', index=False)

In [39]:
# ----------------------------------------------------------------------------
# 17. SUMÁRIO FINAL
# ----------------------------------------------------------------------------
print(f"""VALIDAÇÃO:
   Random Forest:           Accuracy = {acc_rf:.4f}, F1 = {f1_rf:.4f}
   Hist Gradient Boosting:  Accuracy = {acc_gb:.4f}, F1 = {f1_gb:.4f}
""")

VALIDAÇÃO:
   Random Forest:           Accuracy = 0.7681, F1 = 0.7659
   Hist Gradient Boosting:  Accuracy = 0.8136, F1 = 0.8131



In [40]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Usa o teu X_train e y_train (antes do split)
# e o teu melhor modelo (rf_model)
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=2020)

scores = cross_val_score(rf_model, X_train, y_train, 
                         cv=cv_strategy, 
                         scoring='f1_weighted', 
                         n_jobs=-1)

print(f"F1 Scores por Fold: {scores}")
print(f"Média F1: {np.mean(scores):.4f}")
print(f"Desvio Padrão F1: {np.std(scores):.4f}")

F1 Scores por Fold: [0.78646075 0.76119779 0.7763169  0.734348   0.74072506]
Média F1: 0.7598
Desvio Padrão F1: 0.0200
